# 🔗 Multi-Touch Attribution (MTA) Model
**Author:** Vincent Lepore | **Data Source:** GA4 BigQuery Export

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/marketing-analytics-portfolio/blob/main/notebooks/02_mta_model.ipynb)

---

## What is MTA?
Multi-Touch Attribution answers: **"Which marketing channels deserve credit for a conversion?"**

A customer might see a Facebook ad, click a Google search ad two days later, open an email, then convert via direct — which channel gets credit?

### Models in this notebook:
| Model | Description | Best Use Case |
|-------|-------------|---------------|
| **First Touch** | 100% to first channel | Brand awareness measurement |
| **Last Touch** | 100% to last channel | Direct response (Google Ads default) |
| **Linear** | Equal credit | Simple, fair baseline |
| **Time Decay** | More credit closer to conversion | Short sales cycles |
| **U-Shape** | 40/20/40 first/mid/last | Balanced awareness + conversion |
| **Markov Chain** | Transition probabilities + removal effect | Captures journey dependencies |
| **Shapley Value** | Game-theoretic marginal contribution | ⭐ Industry gold standard |

> **Used by:** Rockerbox, Northbeam, Triple Whale, Adobe Analytics

## 1. Setup

In [ ]:
!pip install numpy pandas matplotlib seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict
from itertools import combinations
import math, warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.style.use('dark_background')
COLORS = {'Paid Search':'#00b4d8','Paid Social':'#9b5de5','Email':'#00f5d4',
          'Direct':'#fee440','Affiliate':'#f15bb5','Organic Search':'#00bbf9','Other':'#adb5bd'}

print("✓ Setup complete")

## 2. Load GA4 Data & Reconstruct Customer Journeys

In [ ]:
# Load GA4 events (run notebook 01 first, or load from repo)
try:
    df_ga4 = pd.read_csv("data/raw/ga4_events.csv")
except FileNotFoundError:
    # Download from repo if running fresh in Colab
    import urllib.request
    os.makedirs("data/raw", exist_ok=True)
    print("Note: Run 01_data_generation.ipynb first to generate data")
    raise

# Map UTM source → marketing channel
SOURCE_TO_CHANNEL = {
    'google': 'Paid Search', 'bing': 'Paid Search',
    'meta': 'Paid Social',   'linkedin': 'Paid Social',
    'email': 'Email',        'direct': 'Direct',
    'organic': 'Organic Search', 'affiliate': 'Affiliate',
}

# Reconstruct session-level data
sessions = df_ga4.groupby(['user_pseudo_id', 'ep.session_id'] if 'ep.session_id' in df_ga4.columns 
                           else ['user_pseudo_id']).agg(
    source    = ('traffic_source.source', 'first'),
    ts        = ('event_timestamp', 'min'),
    converted = ('event_name', lambda x: 'purchase' in x.values),
    revenue   = ('ecommerce.purchase_revenue_in_usd', 'max'),
).reset_index()

sessions['channel'] = sessions['source'].map(SOURCE_TO_CHANNEL).fillna('Other')
sessions['revenue'] = sessions['revenue'].fillna(0)

# Build multi-touch journeys (ordered by timestamp)
journeys = sessions.sort_values('ts').groupby('user_pseudo_id').agg(
    channels  = ('channel', list),
    converted = ('converted', 'any'),
    revenue   = ('revenue', 'max'),
).reset_index()

total_conv = journeys['converted'].sum()
total_rev  = journeys[journeys['converted']]['revenue'].sum()

print(f"Journeys:    {len(journeys):,}")
print(f"Conversions: {total_conv:,}  ({total_conv/len(journeys):.1%})")
print(f"Total Rev:   ${total_rev:,.2f}")
print(f"Avg touches: {journeys['channels'].apply(len).mean():.1f}")

# Distribution of journey lengths
length_dist = journeys['channels'].apply(len).value_counts().sort_index()
print("\nJourney length distribution:")
for length, count in length_dist.items():
    print(f"  {length} touch{'es' if length>1 else ''}: {count:,} ({count/len(journeys):.1%})")

## 3. Rule-Based Attribution Models

In [ ]:
def rule_based_attribution(journeys_df, method='linear'):
    """
    Compute revenue attribution for each channel under a given rule-based method.
    
    Methods:
    - first_touch: 100% to first channel
    - last_touch:  100% to last channel  
    - linear:      Equal split
    - time_decay:  Exponential decay toward conversion
    - u_shape:     40/20/40 positional weighting
    """
    credits = defaultdict(float)
    
    for _, row in journeys_df[journeys_df['converted']].iterrows():
        path = row['channels']
        n    = len(path)
        rev  = row['revenue']
        
        if method == 'first_touch':
            weights = [1.0] + [0.0] * (n-1)
        elif method == 'last_touch':
            weights = [0.0] * (n-1) + [1.0]
        elif method == 'linear':
            weights = [1/n] * n
        elif method == 'time_decay':
            raw = [0.5 ** (n-i-1) for i in range(n)]
            weights = [r/sum(raw) for r in raw]
        elif method == 'u_shape':
            if n == 1:   weights = [1.0]
            elif n == 2: weights = [0.5, 0.5]
            else:
                mid = 0.20 / (n-2)
                weights = [0.40] + [mid]*(n-2) + [0.40]
        
        for ch, w in zip(path, weights):
            credits[ch] += w * rev
    
    return dict(credits)

METHODS = ['first_touch','last_touch','linear','time_decay','u_shape']
rule_results = {m: rule_based_attribution(journeys, m) for m in METHODS}

# Display comparison
channels = sorted(set(ch for r in rule_results.values() for ch in r.keys()))
df_compare = pd.DataFrame({
    m.replace('_',' ').title(): [rule_results[m].get(ch,0)/1000 for ch in channels]
    for m in METHODS
}, index=channels)

print("Attribution Revenue by Method ($K):")
print(df_compare.round(1).to_string())

## 4. Markov Chain Attribution

In [ ]:
def build_markov_chain_mta(journeys_df, n_sim=5000):
    """
    Markov Chain Attribution:
    1. Build transition probability matrix from all journeys
    2. Simulate conversion probability from 'Start' node
    3. For each channel: compute removal effect = baseline CVR - CVR without channel
    4. Normalize removal effects → attribution weights
    
    This captures path DEPENDENCIES that rule-based models miss.
    E.g., if Email always appears after Paid Social and drives most conversions,
    Markov correctly credits Paid Social for initiating the high-CVR sequence.
    """
    transitions = defaultdict(lambda: defaultdict(int))
    for _, row in journeys_df.iterrows():
        path = ['Start'] + row['channels'] + (['Conversion'] if row['converted'] else ['Null'])
        for i in range(len(path)-1):
            transitions[path[i]][path[i+1]] += 1
    
    trans_prob = {
        state: {s: c/sum(nexts.values()) for s, c in nexts.items()}
        for state, nexts in transitions.items()
    }
    
    def simulate_cvr(blocked=None):
        conv = 0
        for _ in range(n_sim):
            state = 'Start'
            for _ in range(20):
                if state in ('Conversion','Null'): break
                if state == blocked: state = 'Null'; break
                probs = trans_prob.get(state, {'Null':1.0})
                state = np.random.choice(list(probs.keys()), p=list(probs.values()))
            conv += (state == 'Conversion')
        return conv / n_sim
    
    baseline_cvr = simulate_cvr()
    channel_states = [s for s in trans_prob if s not in ('Start','Conversion','Null')]
    
    removal_effects = {ch: max(0, baseline_cvr - simulate_cvr(ch)) for ch in channel_states}
    total = sum(removal_effects.values()) or 1
    weights = {ch: v/total for ch, v in removal_effects.items()}
    
    return weights, removal_effects, baseline_cvr

print("Building Markov Chain model (this takes ~30 seconds)...")
markov_weights, removal_effects, baseline_cvr = build_markov_chain_mta(journeys)

print(f"\nBaseline conversion rate: {baseline_cvr:.3f}")
print(f"\nRemoval Effects (how much CVR drops when channel is removed):")
for ch, effect in sorted(removal_effects.items(), key=lambda x: -x[1]):
    print(f"  {ch:<20} removal effect: {effect:.4f}  →  attribution: {markov_weights.get(ch,0):.3f}")

## 5. Shapley Value Attribution

In [ ]:
def shapley_mta_monte_carlo(journeys_df, channels, n_perm=500):
    """
    Shapley Value Attribution — Game Theory Approach
    
    Core concept: treat each marketing channel as a 'player' in a cooperative game.
    The 'value' of a coalition = conversion rate of journeys containing those channels.
    
    Shapley value = average marginal contribution across all orderings (permutations).
    
    Why it's the gold standard:
    ✓ Satisfies efficiency, symmetry, dummy, and additivity axioms
    ✓ Uniquely fair credit distribution
    ✓ Handles correlated channels correctly
    
    Computational trick: Monte Carlo sampling of permutations (exact = O(n!) too slow)
    """
    journey_sets = [
        (frozenset(row['channels']), int(row['converted']), float(row['revenue']))
        for _, row in journeys_df.iterrows()
    ]
    
    cache = {}
    def v(subset):
        key = frozenset(subset)
        if key not in cache:
            if not key:
                cache[key] = sum(cv for _,cv,_ in journey_sets) / max(len(journey_sets), 1)
            else:
                relevant = [cv for cs,cv,_ in journey_sets if key.issubset(cs)]
                cache[key] = sum(relevant)/len(relevant) if relevant else 0.0
        return cache[key]
    
    channels = [c for c in channels if any(c in cs for cs,_,_ in journey_sets)]
    n = len(channels)
    shapley = defaultdict(float)
    rng = np.random.default_rng(42)
    
    for _ in range(n_perm):
        perm = rng.permutation(n)
        coalition, v_prev = [], v([])
        for idx in perm:
            ch = channels[idx]
            coalition.append(ch)
            v_new = v(coalition)
            shapley[ch] += (v_new - v_prev)
            v_prev = v_new
    
    for ch in channels: shapley[ch] /= n_perm
    
    # Shift to non-negative and normalize
    min_sv = min(shapley.values())
    shifted = {ch: v - min_sv for ch, v in shapley.items()}
    total = sum(shifted.values()) or 1
    return {ch: v/total for ch, v in shifted.items()}, dict(shapley)

CHANNELS = list(set(ch for path in journeys['channels'] for ch in path))
print(f"Computing Shapley values for {len(CHANNELS)} channels with 500 Monte Carlo samples...")

shapley_norm, shapley_raw = shapley_mta_monte_carlo(journeys.sample(2000, random_state=42), CHANNELS)

print("\nShapley Values (normalized attribution weights):")
for ch, sv in sorted(shapley_norm.items(), key=lambda x: -x[1]):
    bar = '█' * int(sv * 40)
    print(f"  {ch:<20} {sv:.3f}  {bar}")

## 6. Visualization — All Models Compared

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0d1117')

# Left: Multi-model comparison bar chart
all_methods = {
    'First Touch':  rule_results['first_touch'],
    'Last Touch':   rule_results['last_touch'],
    'Linear':       rule_results['linear'],
    'Time Decay':   rule_results['time_decay'],
    'U-Shape':      rule_results['u_shape'],
    'Markov':       {ch: w * total_rev for ch, w in markov_weights.items()},
    'Shapley':      {ch: w * total_rev for ch, w in shapley_norm.items()},
}

channels_ordered = sorted(CHANNELS, key=lambda c: -shapley_norm.get(c, 0))
method_colors = ['#e63946','#f4a261','#2a9d8f','#457b9d','#a8dadc','#9b5de5','#00b4d8']

x = np.arange(len(channels_ordered))
width = 0.12
for i, (method, results) in enumerate(all_methods.items()):
    vals = [results.get(ch, 0)/1000 for ch in channels_ordered]
    axes[0].bar(x + i*width - 3*width, vals, width, label=method, 
                color=method_colors[i], alpha=0.85, edgecolor='none')

axes[0].set_xticks(x)
axes[0].set_xticklabels(channels_ordered, rotation=25, ha='right', fontsize=9)
axes[0].set_ylabel('Attributed Revenue ($K)', fontsize=10)
axes[0].set_title('MTA Model Comparison — All 7 Methods', fontsize=12, fontweight='bold', pad=15)
axes[0].legend(fontsize=8, loc='upper right')
axes[0].set_facecolor('#161b22')
axes[0].grid(axis='y', alpha=0.3, color='white')

# Right: Shapley vs Last Touch delta (shows bias of last-touch)
shapley_rev = {ch: shapley_norm.get(ch,0) * total_rev for ch in channels_ordered}
last_touch_rev = rule_results['last_touch']
deltas = [(shapley_rev.get(ch,0) - last_touch_rev.get(ch,0))/1000 for ch in channels_ordered]
bar_colors = ['#00f5d4' if d >= 0 else '#f15bb5' for d in deltas]
axes[1].barh(channels_ordered, deltas, color=bar_colors, alpha=0.85, edgecolor='none')
axes[1].axvline(0, color='white', linewidth=0.8, alpha=0.5)
axes[1].set_xlabel('Revenue Difference ($K)', fontsize=10)
axes[1].set_title('Shapley vs Last Touch — Attribution Bias', fontsize=12, fontweight='bold', pad=15)
axes[1].set_facecolor('#161b22')
axes[1].grid(axis='x', alpha=0.3, color='white')

green_patch = mpatches.Patch(color='#00f5d4', label='Under-credited by Last Touch')
red_patch   = mpatches.Patch(color='#f15bb5', label='Over-credited by Last Touch')
axes[1].legend(handles=[green_patch, red_patch], fontsize=9)

plt.tight_layout(pad=3)
plt.savefig('data/processed/mta_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("\n💡 Key insight: Last Touch over-credits conversion channels (Direct, Paid Search)")
print("   Shapley/Markov correctly credit awareness channels that initiated the journey.")